In [28]:
%pip install -e /Users/instafiore/Workspace/AMOSUM

Obtaining file:///Users/instafiore/Workspace/AMOSUM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for amosum (pyproject.toml) ... done
  Created wheel for amosum: filename=amosum-0.0.1-py3-none-any.whl size=3314 sha256=104ad7b9d3a7f3cf1c0bf8ca792fb96a1d1311a8e89de18061e55ba4a5f170ec
  Stored in directory: /private/var/folders/60/msx2m7995xv41v59mx28gt5w0000gn/T/pip-ephem-wheel-cache-rn6ieob_/wheels/f9/7b/3a/a109cecea02e5123eee232e700087c069a0eb849a8a3ec42cb
Successfully built amosum
  Attempting uninstall: amosum
    Found existing installation: amosum 0.0.1
    Uninstalling amosum-0.0.1:
      Successfully uninstalled amosum-0.0.1
Note: you may need to restart the kernel to use updated packages.


In [29]:
import argparse
import sys
import os
import subprocess
from typing import Dict, List, Set
import re


from amosum.utility import *
from amosum.amoclingo.propagator_clingo_c.runner_clingo import *
from amosum.amoclingo.propagator_clingo_py.runner_clingo import *
from amosum.amowasp.propagator_wasp_py.runner_wasp import *
import sys



In [30]:
class Checker:
    def __init__(self, args: Dict):
        solver = args.get("lang")
        language = args.get("lang")
        self.runner : RunnerClingoCpp | RunnerClingoPython
        if language == "py":
            if solver == "clingo":
                self.runner = RunnerClingoPython(parameters=args)
            else:
                self.runner = RunnerWasp(parameters=args)
        elif language == "cpp":
            self.runner = RunnerClingoCpp(parameters=args)
        self.runner = RunnerClingoCpp(args)
        self.args = args

    def check_correctness(self) -> bool:
        """
        Checks if each answer set produced by the propagator is also produced by clingo.

        Returns:
        bool: True if all propagator's answer sets are found in clingo's answer sets, False otherwise.
        """
        # Run the propagator
        self.runner.printOutput = False
        results: List[Result] = self.runner.run()
        def notAux(x: str):
            return re.search("^__", x) is None
        propagator_answer_sets = [set(filter(notAux, r.model.assigment)) for r in results] if results[0].exitCode != 20 else "UNSAT"
        if propagator_answer_sets == "UNSAT":
            print("Propagator found UNSAT")
            propagator_answer_sets = []


        # Run clingo
        clingo_answer_sets = run_clingo(self.args)
        if clingo_answer_sets == "UNSAT":
            print("Clingo found UNSAT")
            clingo_answer_sets = []

        correct = True 
        # Compare answer sets
        for pas in propagator_answer_sets:
            if pas not in clingo_answer_sets:
                print(f" => Answer set {pas} from propagator not found in clingo's answer sets.")
                correct = False
                break
        
        for pas in clingo_answer_sets:
            if pas not in propagator_answer_sets:
                print(f" <= Answer set {pas} from clingo not found in propagator's answer sets.")
                correct = False
                break

        if not correct:
            # debug("propagator_answer_sets: ")
            # for ans in propagator_answer_sets:
            #     debug(ans)
            # debug("clingo_answer_sets: ")
            # for ans in clingo_answer_sets:
            #     debug(ans)
            pass
        
        return correct


In [31]:
def check():
    args = parse_args(checkCorrectness=True)

    if args["check"] == "all":
        print(f"[python] parameters: {args}")
        args["num_models"] = 0
        args["check"] = False
        checker = Checker(args)
        success = checker.check_correctness()
    # elif args["check"] == "unsat":
    #     success = checkUnsatFromResultFile(args["unsatfile"])
    #     s = "Passed " if success else "Failed"
    #     print(f"{s} check unsat for file: {args['unsatfile']}")
    # elif args["check"] == "unsat-clown":
    #     success = check_unsatisfability_clown(args["unsatfile"])
    #     s = "Passed " if success else "Failed"
    #     print(f"{s} check unsat for file: {args['unsatfile']}")
    # else:
    #     success = check(args["encoding"],args["instance"],args["file_answerset"])
    #     s = "Passed " if success else "Failed"
    #     e = args["encoding"]
    #     i = args["instance"]
    #     f = args["file_answerset"]
    #     print(f"{s} check correctness for encoding: {e} instance: {i} file: {f}")
    return success

            
import sys
import os
import re

In [33]:
# TOY
import os
import glob

encodings = [e for e in glob.glob("bench/kn_toy/*") if re.search("encoding-amosum-", e)]
instances = [e for e in glob.glob("bench/kn_toy/instances/*")]
print(encodings)
success = True
for e in encodings:
    for i in instances:
        sys.argv = [sys.argv[0]]
        sys.argv.append(f"-e={e}")
        sys.argv.append(f"-i={i}")
        sys.argv.append(f"-n=0")
        sys.argv.append("all")
        success = check()
        if not success:
            break
print("[ALL] Check passed 🔥" if success else "[ALL] Check failed 🚨")

['bench/kn_toy/encoding-amosum-eo.asp', 'bench/kn_toy/encoding-amosum-amo.asp']
[python] parameters: {'encoding': 'bench/kn_toy/encoding-amosum-eo.asp', 'instance': 'bench/kn_toy/instances/toy_le_4.asp', 'lazy': 'false', 'reason': 'nomin', 'lang': 'cpp', 'models': 0, 'log_file': 'log', 'static_mpc': 'false', 'check': 'all', 'solver': 'clingo'}
encoding: bench/kn_toy/encoding-amosum-eo.asp
instance: bench/kn_toy/instances/toy_le_4.asp
run:	 /Users/instafiore/Workspace/AMOSUM/amosum/amoclingo/propagator_clingo_c/bin/./amosum_cpp            -encoding=/tmp/.encoding-amosum-eo_without_amosum_2026-05-31-00-11-27-645764.asp            -instance=/tmp/.toy_le_4_without_amosum_2026-05-31-00-11-27-694626.asp             -models=0             -logfile=log             -serialize -amosum_propagators="ge_eo -id (1,lb(9,1)) -encoding bench/kn_toy/encoding-amosum-eo.asp -instance bench/kn_toy/instances/toy_le_4.asp -lazy false -reason nomin -lang cpp -models 0 -log_file log -static_mpc false -check Fal